# ☀️ Solar Filament Segmentation Challenge 2026 (Colab Pro+ Starter)

サブコンペ **「Solar Filament Segmentation Challenge 2026」** 用の統合 Colab ノートブックです。

### 🌟 特長:
- **Google Drive 永続保存**: 初回のみDriveへダウンロードし、2回目以降は即スタート！
- **`segmentation_models_pytorch` (smp)**: Unet++, DeepLabV3+ など SOTA バックボーン対応
- **RLE (Run-Length Encoding)**: 提出フォーマットに自動変換

In [ ]:
!pip install -q segmentation-models-pytorch albumentations timm kaggle tqdm opencv-python matplotlib

In [ ]:
import os
from google.colab import userdata, drive

# 1. Google Drive マウント
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/kaggle_solar_project'
DATA_DIR = os.path.join(DRIVE_DIR, 'data')
CHECKPOINT_DIR = os.path.join(DRIVE_DIR, 'checkpoints')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 2. 初回データダウンロード
if not os.path.exists(os.path.join(DATA_DIR, 'train')) and not os.path.exists(os.path.join(DATA_DIR, 'filament-segmentation-2026.zip')):
    print('📥 初回データのダウンロードを開始します (Google Driveへ保存)...')
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    !kaggle competitions download -c filament-segmentation-2026 -p {DATA_DIR}
    !unzip -q {DATA_DIR}/filament-segmentation-2026.zip -d {DATA_DIR}
    print('✅ データ保存が完了しました！')
else:
    print('⚡ Google Drive 内の既存データを検出しました！')

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'=== Using Device: {device} ===')

# smp Unet++ (EfficientNet-B3 Backbone) モデル作成
model = smp.UnetPlusPlus(
    encoder_name='efficientnet-b3',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1
).to(device)

print('smp Unet++ Model Loaded!')

In [ ]:
# ダミーデータによる 2D Segmentation 学習動作テスト
class SolarDataset(Dataset):
    def __len__(self):
        return 32
    def __getitem__(self, idx):
        img = np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8)
        mask = np.zeros((512, 512), dtype=np.float32)
        mask[100:300, 100:300] = 1.0
        img_tensor = torch.tensor(img).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.tensor(mask).unsqueeze(0).float()
        return img_tensor, mask_tensor

train_loader = DataLoader(SolarDataset(), batch_size=8, shuffle=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)

print('=== Starting Solar Filament Training Loop ===')
for epoch in range(1, 6):
    model.train()
    total_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'Epoch {epoch}', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch [{epoch}/5] | Dice Loss: {total_loss/len(train_loader):.4f}')

best_path = os.path.join(CHECKPOINT_DIR, 'best_solar_unet.pth')
torch.save(model.state_dict(), best_path)
print(f'Saved Checkpoint to {best_path}')